# Optimization Showcase — defense preview (Manhattan toy)

A **slide-ready walkthrough of one memetic optimization loop** on the Manhattan toy city, then the
**whole optimization run** — stitched from the pieces the thesis already uses. The grid is deliberately
simple so each mechanism is visually unambiguous. Beats: simulation, candidates + fitness, pheromone,
gap, hub crossover, local search, and finally the full multi-generation evolution.

> A companion notebook, `nb_optimization_showcase_iligan.ipynb`, runs the same pipeline on the **real
> Iligan network** and animates the real 30-generation production run. Use the toy here for **clarity of
> mechanism**; use Iligan for **realism + breadth**. One knob: `smoke=True` (fast) → `smoke=False` (slow).

In [ ]:
import showcase_optimization as S
from IPython.display import Image, display

# ── the one knob ──  set smoke=False for slide-quality (slow); keep True to sanity-check fast
cfg = S.make_config(smoke=True, city="toy")

env = S.setup_env(cfg)
members = S.build_population(env, cfg)   # generate + simulate the candidate route systems
print(f"{'SMOKE' if cfg['smoke'] else 'FULL'} toy  |  {len(members)} candidates, "
      f"{cfg['sim_ticks']}-tick sims  |  best F_sim = {members[0]['fsim']:.0f}  |  -> {cfg['out_dir']}/")

## Beat 1 — the simulation we are optimizing
`Jeep` agents run their loops while `Passenger` agents spawn, walk, wait, board (under capacity) and ride. The number on each jeep is its onboard count; the dashboard tracks live tick / active / done. Captured every *N*-th tick.

In [ ]:
_, frames = S.simulate_with_frames(env, members[0]["routes"], cfg)
gif = S.save_gif(frames, S._p(cfg, "01_simulation.gif"), cfg["gif_ms"])
print(f"{len(frames)} frames"); display(Image(gif))

## Beat 2 — candidate route systems and their fitness
The optimizer generates many route systems and scores each by **simulated Total User Cost** ($F_{sim}$, lower is better). These are the raw material the genetic operators recombine.

In [ ]:
display(Image(S.render_population_grid(env, members, "routes", S._p(cfg, "02_route_systems.png"), cfg["dpi"])))

## Beat 3 — realized demand memory (pheromone $\tau$)
Each simulation leaves a **stigmergic pheromone trace** on the edges passengers actually flowed through — the network's realized demand memory. Brighter = more realized travel.

In [ ]:
display(Image(S.render_population_grid(env, members, "pheromone", S._p(cfg, "03_pheromones.png"), cfg["dpi"])))

## Beat 4 — demand-service gap
Realized demand vs fleet supply gives the **Proportional Demand-Service Gap** ($\Delta = P-S$): red corridors *underserved*, blue *oversupplied*. This signed field steers the local search in Beat 6; $D(R)=\sum|\Delta|$ summarizes the mismatch.

In [ ]:
display(Image(S.render_population_grid(env, members, "gap", S._p(cfg, "04_demand_service_gaps.png"), cfg["dpi"])))

## Beat 5 — Topological Hub Crossover (memetic recombination)
The two fittest candidates become parents. The **topological hub** (top-decile pheromone corridors) of the fitter parent is preserved as the child's **trunk** (red); non-conflicting routes from the other parent are grafted on as **feeders** (green), plus a fitness-weighted pheromone blend. We keep the better of the two parent orderings.

In [ ]:
from fig_memetic import fig_hub_crossover, fig_pheromone_blend
scene = S.build_crossover_scene(env, members, cfg)
s = scene["stats"]
print(f"child F_sim = {s['child_fsim']:.0f}   (parents {s['A_fsim']:.0f}, {s['B_fsim']:.0f})")
display(Image(fig_hub_crossover(scene, S._p(cfg, "05_hub_crossover.png"))))
display(Image(fig_pheromone_blend(scene, S._p(cfg, "05b_pheromone_blend.png"))))

## Beat 6 — Lamarckian local search (\"mutation\") at bumped intensity
The three operators refine the child along the demand-service gap. At default intensity they barely perturb the network, so we **bump the intensity** (`mut_intensity`) to make the move unmistakable. We run all three and keep the best re-simulated result; touched routes are highlighted.

In [ ]:
child = scene["child"]
mut = S.apply_obvious_mutation(env, child.routes, child.pheromones, cfg, base_cost=child.cost)
mut["base_cost"] = child.cost
print(f"{mut['op']}: {mut['n_changed']} edges changed   F_sim {child.cost:.0f} -> {mut.get('after_cost', float('nan')):.0f}")
display(Image(S.render_mutation(env, mut, S._p(cfg, "06_local_search.png"), cfg["dpi"])))

## Beat 7 — the full optimization (feel the breadth)
The beats above are one loop. Here is the **whole run**: a generation-by-generation **animation** — the best network, its realized demand memory $\tau$, its demand-service gap, and the convergence curve (best in red, population mean in blue) all advancing together — plus the per-generation storyboard and the convergence curve.

> **Scale note.** At `smoke=True` the toy run is short, so the animation is brief and nearly flat. Set `smoke=False` for a real multi-generation descent. The Iligan companion animates the **real 30-generation production run** — that is the breadth slide to show.

In [ ]:
paths = S.run_optimization_views(cfg)    # convergence curve + per-gen storyboard + evolution ANIMATION
if "evolution" in paths:                  # the generation-by-generation animation is the centerpiece
    display(Image(paths["evolution"]))
for k in ("07_convergence", "07b_evolution"):
    if k in paths:
        display(Image(paths[k]))

---
### Producing the slide-quality set
Re-run with **`cfg = S.make_config(smoke=False)`** on a strong machine; for a richer Beat-7 evolution bump
`S.make_config(smoke=False, preview_generations=30, preview_population=20)`. Figures save under
`cfg['out_dir']` at `cfg['dpi']`. One-shot: `python showcase_optimization.py --full`.